In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, PathPatch
import matplotlib as mpl
import numpy as np 
import pandas as pd
import dash
from dash import html, dcc, Input, Output
import plotly.express as px
import os


Import the data into a data frams

In [ ]:
filename = 'finaldata1-3_Youllee_update_20260119.csv'
df = pd.read_csv(filename)

In [ ]:
if 'gender' in df.columns:
    df['gender_label'] = df['gender'].map({1: 'Male', 2: 'Female'}).fillna('Other')
else:
    df['gender_label'] = 'Unknown'

# --- App Initialization ---
app = dash.Dash(__name__)
server = app.server  # Expose the server variable for deployments (e.g., Gunicorn)

# --- App Layout ---
app.layout = html.Div([
    html.H1("Survey Data Interactive Dashboard", style={'textAlign': 'center', 'marginBottom': '30px'}),

    # Control Panel Row
    html.Div([
        # X-Axis Dropdown
        html.Div([
            html.Label("Select X-Axis Variable:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='xaxis-column',
                options=[
                    {'label': 'Age', 'value': 'age'},
                    {'label': 'Socioeconomic Status (SES)', 'value': 'ses'},
                    {'label': 'Political Orientation', 'value': 'poli'},
                    {'label': 'Affective Polarization (Wave 1)', 'value': 'affectpol1'},
                ],
                value='age',
                clearable=False
            )
        ], style={'width': '48%', 'display': 'inline-block', 'verticalAlign': 'top'}),

        # Y-Axis Dropdown
        html.Div([
            html.Label("Select Y-Axis Variable:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='yaxis-column',
                options=[
                    {'label': 'Affective Polarization (Wave 1)', 'value': 'affectpol1'},
                    {'label': 'Perceived Polarization (Lay) (Wave 1)', 'value': 'perpolarlay1'},
                    {'label': 'Perceived Polarization (Elite) (Wave 1)', 'value': 'perpolarelite1'},
                    {'label': 'Civic Engagement (Wave 1)', 'value': 'civicengage1'},
                    {'label': 'Vote Intention (Wave 2)', 'value': 'voteintention2'},
                ],
                value='affectpol1',
                clearable=False
            )
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block', 'verticalAlign': 'top'})
    ], style={'padding': '20px', 'backgroundColor': '#f9f9f9', 'borderRadius': '5px'}),

    # Graph Component
    dcc.Graph(id='scatter-plot', style={'marginTop': '20px'}),

    # Filter Row
    html.Div([
        html.Label("Filter by Gender:", style={'fontWeight': 'bold', 'marginRight': '10px'}),
        dcc.Checklist(
            id='gender-filter',
            options=[
                {'label': ' Male', 'value': 'Male'},
                {'label': ' Female', 'value': 'Female'}
            ],
            value=['Male', 'Female'],
            inline=True,
            inputStyle={"margin-right": "5px", "margin-left": "10px"}
        )
    ], style={'textAlign': 'center', 'padding': '20px'})
])

# --- Callbacks ---
@app.callback(
    Output('scatter-plot', 'figure'),
    [Input('xaxis-column', 'value'),
     Input('yaxis-column', 'value'),
     Input('gender-filter', 'value')]
)
def update_graph(xaxis_col, yaxis_col, selected_genders):
    # Filter the dataframe based on selected genders
    filtered_df = df[df['gender_label'].isin(selected_genders)]

    # Create the scatter plot using Plotly Express
    fig = px.scatter(
        filtered_df,
        x=xaxis_col,
        y=yaxis_col,
        color='gender_label',
        hover_data=['ID', 'age', 'gender'],
        title=f"Correlation: {xaxis_col} vs {yaxis_col}",
        labels={
            'age': 'Age',
            'ses': 'SES',
            'poli': 'Political Orientation',
            'affectpol1': 'Affective Polarization',
            'gender_label': 'Gender'
        },
        opacity=0.7
    )

    fig.update_layout(
        margin={'l': 40, 'b': 40, 't': 40, 'r': 0},
        hovermode='closest',
        legend_title_text='Gender'
    )

    return fig

# --- Main Execution ---
if __name__ == '__main__':
    app.run(debug=True)

ObsoleteAttributeException: app.run_server has been replaced by app.run